# RQ3.2 — Prompt refinement

Analyzes the four prompt-refinement factors:

1. Factor 1 — Class-definition enrichment
2. Factor 2 — Heuristic guidance for ambiguous commits
3. Factor 3 — Reasoning and example-based prompting
4. Factor 4 — Confidence-based abstention

Inputs: `results/rq3_2/metrics/factor_1_macro_f1.csv` to `factor_4_macro_f1.csv`. The treatments analyzed for each factor are declared in `FACTORS` (first code cell); other columns in the CSVs are reported and ignored.

Outputs (displayed inline; no files are written): per-factor descriptive statistics, normality and omnibus tests, Holm-corrected pairwise comparisons, and a cross-factor summary table.

If the CSVs are not found automatically, set `DATA_ROOT_OVERRIDE` in the first code cell to `results/rq3_2` (or another directory containing `metrics/`).

Run: *Run All*.

In [1]:
from pathlib import Path
from itertools import combinations
import importlib.util

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import friedmanchisquare

if importlib.util.find_spec("pingouin") is None:
    raise ImportError(
        "This notebook requires pingouin. Install the "
        "requirements before running it (pip install -r requirements.txt)."
    )

import pingouin as pg

ALPHA = 0.05
EXPECTED_PROJECTS = 30
SUBJECT = "project"

# Optional: set this to results/rq3_2 (or another directory containing
# the metrics/ folder) if automatic path resolution does not find the CSVs.
DATA_ROOT_OVERRIDE = None

LABELS = {
    "p0_baseline": "Zero-shot baseline",
    "p1_definition_augmented": "Literature definitions",
    "p2_angular_definition": "Angular definitions",
    "p3_extensive_definition": "Extensive definitions",
    "p4_mix_definition": "Hybrid definitions",
    "p_negative_constraints": "Negative constraints",
    "p_chore_rule_and_negative": "Chore + negative constraints",
    "p_chore_rule": "Chore recovery",
    "p_file_pattern": "File-pattern rules",
    "p_priority_rules": "Priority rules",
    "p_chore_negative_and_dominant": "Chore + negative + dominant intent",
    "p_header_structured": "Structured headers",
    "p_code_disambiguation": "Code disambiguation",
    "p_infra_disambiguation": "Infrastructure disambiguation",
    "p_disambiguation": "Code + infrastructure disambiguation",
    "p_zero-shot_cot": "Zero-shot chain-of-thought",
    "p_few-shot": "Few-shot",
    "p_cot_few-shot": "Few-shot + chain-of-thought",
    "p_exclude_low": "Exclude low confidence",
    "p_high_only": "High confidence only",
}

FACTORS = {
    "factor_1_definitions": {
        "title": "Factor 1 — Class-definition enrichment",
        "reference": "p0_baseline",
        "treatments": [
            "p0_baseline",
            "p1_definition_augmented",
            "p2_angular_definition",
            "p3_extensive_definition",
            "p4_mix_definition",
        ],
        "expected_factor_value": "factor_1_definitions",
        "paths": [
            "../metrics/factor_1_macro_f1.csv",
            "metrics/factor_1_macro_f1.csv",
            "rq3_2/metrics/factor_1_macro_f1.csv",
            "results/rq3_2/metrics/factor_1_macro_f1.csv",
        ],
    },
    "factor_2_heuristics": {
        "title": "Factor 2 — Heuristic guidance for ambiguous commits",
        "reference": "p0_baseline",
        "treatments": [
            "p0_baseline",
            "p2_angular_definition",
            "p_negative_constraints",
            "p_chore_rule_and_negative",
            "p_chore_rule",
            "p_file_pattern",
            "p_priority_rules",
            "p_chore_negative_and_dominant",
            "p_header_structured",
            "p_code_disambiguation",
            "p_infra_disambiguation",
            "p_disambiguation",
        ],
        "expected_factor_value": "factor_2_heuristics",
        "paths": [
            "../metrics/factor_2_macro_f1.csv",
            "metrics/factor_2_macro_f1.csv",
            "rq3_2/metrics/factor_2_macro_f1.csv",
            "results/rq3_2/metrics/factor_2_macro_f1.csv",
        ],
    },
    "factor_3_few_shot_cot": {
        "title": "Factor 3 — Reasoning and example-based prompting",
        "reference": "p_code_disambiguation",
        "treatments": [
            "p0_baseline",
            "p_code_disambiguation",
            "p_zero-shot_cot",
            "p_few-shot",
            "p_cot_few-shot",
        ],
        "expected_factor_value": "factor_3_few_shot_cot",
        "paths": [
            "../metrics/factor_3_macro_f1.csv",
            "metrics/factor_3_macro_f1.csv",
            "rq3_2/metrics/factor_3_macro_f1.csv",
            "results/rq3_2/metrics/factor_3_macro_f1.csv",
        ],
    },
    "factor_4_abstention": {
        "title": "Factor 4 — Confidence-based abstention",
        "reference": "p_code_disambiguation",
        "treatments": [
            "p0_baseline",
            "p_code_disambiguation",
            "p_exclude_low",
            "p_high_only",
        ],
        "expected_factor_value": "factor_4_abstention",
        "paths": [
            "../metrics/factor_4_macro_f1.csv",
            "metrics/factor_4_macro_f1.csv",
            "rq3_2/metrics/factor_4_macro_f1.csv",
            "results/rq3_2/metrics/factor_4_macro_f1.csv",
        ],
    },
}

def _search_roots():
    roots = []
    if DATA_ROOT_OVERRIDE is not None:
        roots.append(Path(DATA_ROOT_OVERRIDE).expanduser().resolve())

    cwd = Path.cwd().resolve()
    roots.append(cwd)
    roots.extend(list(cwd.parents)[:5])

    unique = []
    seen = set()
    for root in roots:
        marker = str(root)
        if marker not in seen:
            unique.append(root)
            seen.add(marker)
    return unique


def resolve_factor_file(factor_key):
    cfg = FACTORS[factor_key]
    attempted = []

    for root in _search_roots():
        for relative in cfg["paths"]:
            candidate = (root / relative).resolve()
            attempted.append(candidate)
            if candidate.is_file():
                return candidate

    formatted = "\n".join(f"  - {path}" for path in attempted)
    raise FileNotFoundError(
        f"Could not locate the Macro-F1 CSV for {factor_key}.\n"
        f"Paths checked:\n{formatted}\n\n"
        "Set DATA_ROOT_OVERRIDE to the directory that contains "
        "the metrics/ folder."
    )


def load_factor_data(factor_key):
    cfg = FACTORS[factor_key]
    path = resolve_factor_file(factor_key)
    df = pd.read_csv(path)

    required = [SUBJECT, *cfg["treatments"]]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(
            f"{factor_key}: missing required columns {missing} in {path}"
        )

    if df[SUBJECT].duplicated().any():
        duplicates = (
            df.loc[df[SUBJECT].duplicated(), SUBJECT]
            .astype(str)
            .tolist()
        )
        raise ValueError(
            f"{factor_key}: each project must occur once. "
            f"Duplicates: {duplicates}"
        )

    if len(df) != EXPECTED_PROJECTS or df[SUBJECT].nunique() != EXPECTED_PROJECTS:
        raise ValueError(
            f"{factor_key}: expected {EXPECTED_PROJECTS} projects, "
            f"found {len(df)} rows and {df[SUBJECT].nunique()} unique projects."
        )

    for treatment in cfg["treatments"]:
        converted = pd.to_numeric(df[treatment], errors="coerce")
        if converted.isna().any():
            bad_projects = df.loc[converted.isna(), SUBJECT].astype(str).tolist()
            raise ValueError(
                f"{factor_key}: non-numeric or missing values in {treatment}; "
                f"projects: {bad_projects}"
            )
        df[treatment] = converted

    if "factor" in df.columns:
        observed = set(df["factor"].dropna().astype(str))
        expected = cfg["expected_factor_value"]
        if observed and observed != {expected}:
            raise ValueError(
                f"{factor_key}: expected factor column value {expected!r}, "
                f"observed {sorted(observed)}"
            )

    analyzed = set(cfg["treatments"])
    ignored_numeric = []
    for column in df.columns:
        if column in analyzed or column in {SUBJECT, "factor"}:
            continue
        converted = pd.to_numeric(df[column], errors="coerce")
        original_non_missing = df[column].notna()
        if (
            original_non_missing.any()
            and converted.loc[original_non_missing].notna().all()
        ):
            ignored_numeric.append(column)

    print(f"Data file: {path}")
    print(f"Projects: {df[SUBJECT].nunique()}")
    print(f"Analyzed treatments ({len(cfg['treatments'])}): {cfg['treatments']}")
    if ignored_numeric:
        print(
            "Extra numeric columns intentionally excluded from this factor: "
            f"{ignored_numeric}"
        )

    columns = ["factor", SUBJECT, *cfg["treatments"]]
    return df[[column for column in columns if column in df.columns]].copy()


## Analysis functions

In [2]:
def _pretty_label(treatment):
    if treatment in LABELS:
        return LABELS[treatment]
    return treatment.replace("_", " ").replace("-", " ").strip().title()


def _run_parametric_route(
    long_data,
    analysis_data,
    subject,
    treatments,
    dv_name,
    alpha,
):
    n_subjects = analysis_data[subject].nunique()
    k = len(treatments)

    sphericity = pg.sphericity(
        data=long_data,
        dv=dv_name,
        within="treatment",
        subject=subject,
    )
    apply_gg = bool(sphericity.pval < alpha)

    anova = pg.rm_anova(
        data=long_data,
        dv=dv_name,
        within="treatment",
        subject=subject,
        correction=True,
        detailed=True,
    )
    effect = anova.iloc[0]

    epsilon = float(effect["eps"])
    df1_uncorrected = k - 1
    df2_uncorrected = (k - 1) * (n_subjects - 1)

    if apply_gg:
        final_p = float(effect["p_GG_corr"])
        final_df1 = epsilon * df1_uncorrected
        final_df2 = epsilon * df2_uncorrected
        correction = "Greenhouse-Geisser"
    else:
        final_p = float(effect["p_unc"])
        final_df1 = float(df1_uncorrected)
        final_df2 = float(df2_uncorrected)
        correction = "None"

    omnibus = {
        "test": "Repeated-measures ANOVA",
        "statistic_name": "F",
        "statistic": float(effect["F"]),
        "df1": final_df1,
        "df2": final_df2,
        "p": final_p,
        "effect_name": "eta_G_squared",
        "effect": float(effect["ng2"]),
        "epsilon": epsilon,
        "correction": correction,
        "significant": final_p < alpha,
    }

    posthoc = None
    if final_p < alpha:
        posthoc = pg.pairwise_tests(
            data=long_data,
            dv=dv_name,
            within="treatment",
            subject=subject,
            parametric=True,
            padjust="holm",
            effsize="hedges",
            return_desc=True,
        )
        posthoc = posthoc[[
            "A", "B", "mean_A", "mean_B", "T", "dof",
            "p_unc", "p_corr", "p_adjust", "hedges",
        ]].copy()
        posthoc["significant"] = posthoc["p_corr"] < alpha

    return {
        "sphericity": sphericity,
        "anova": anova,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def _run_nonparametric_route(
    analysis_data,
    treatments,
    alpha,
):
    n_subjects = len(analysis_data)
    k = len(treatments)

    friedman = friedmanchisquare(
        *[analysis_data[treatment].to_numpy() for treatment in treatments]
    )
    q_statistic = float(friedman.statistic)
    p_value = float(friedman.pvalue)
    kendalls_w = q_statistic / (n_subjects * (k - 1))

    omnibus = {
        "test": "Friedman",
        "statistic_name": "Q",
        "statistic": q_statistic,
        "df1": float(k - 1),
        "df2": None,
        "p": p_value,
        "effect_name": "Kendalls_W",
        "effect": kendalls_w,
        "epsilon": None,
        "correction": "Not applicable",
        "significant": p_value < alpha,
    }

    posthoc = None
    if p_value < alpha:
        rows = []

        for treatment_a, treatment_b in combinations(treatments, 2):
            values_a = analysis_data[treatment_a].to_numpy()
            values_b = analysis_data[treatment_b].to_numpy()

            if np.allclose(values_a - values_b, 0):
                w_value = 0.0
                pair_p = 1.0
                rbc = 0.0
            else:
                wilcoxon = pg.wilcoxon(
                    values_a,
                    values_b,
                    alternative="two-sided",
                ).iloc[0]
                w_value = float(wilcoxon["W-val"])
                pair_p = float(wilcoxon["p-val"])
                rbc = float(wilcoxon["RBC"])

            rows.append({
                "A": treatment_a,
                "B": treatment_b,
                "mean_A": float(np.mean(values_a)),
                "mean_B": float(np.mean(values_b)),
                "W": w_value,
                "p_unc": pair_p,
                "RBC": rbc,
            })

        posthoc = pd.DataFrame(rows)
        reject, corrected_p = pg.multicomp(
            posthoc["p_unc"].to_numpy(),
            alpha=alpha,
            method="holm",
        )
        posthoc["p_corr"] = corrected_p
        posthoc["significant"] = reject

    return {
        "sphericity": None,
        "anova": None,
        "omnibus": omnibus,
        "posthoc": posthoc,
    }


def run_repeated_measures_analysis(
    data,
    subject,
    treatments,
    dv_name="macro_f1",
    alpha=ALPHA,
    reference=None,
):
    if len(treatments) < 3:
        raise ValueError("At least three treatments are required.")

    analysis_data = data[[subject, *treatments]].copy()

    if analysis_data.isna().any().any():
        raise ValueError(
            "The factor contains missing values. The RQ3.2 analysis expects "
            "the same complete set of 30 projects for every treatment."
        )

    long_data = analysis_data.melt(
        id_vars=subject,
        value_vars=treatments,
        var_name="treatment",
        value_name=dv_name,
    )

    descriptive = (
        long_data.groupby("treatment")[dv_name]
        .agg(N="count", Mean="mean", SD="std")
        .sort_values("Mean", ascending=False)
    )

    shapiro_rows = []
    for treatment in treatments:
        normality = pg.normality(
            analysis_data[treatment],
            method="shapiro",
            alpha=alpha,
        ).iloc[0]

        shapiro_rows.append({
            "Treatment": treatment,
            "N": int(analysis_data[treatment].notna().sum()),
            "W": float(normality["W"]),
            "p": float(normality["pval"]),
            "Normality not rejected": bool(normality["normal"]),
        })

    shapiro = pd.DataFrame(shapiro_rows).set_index("Treatment")
    all_normal = bool(shapiro["Normality not rejected"].all())

    if all_normal:
        route = "parametric"
        inferential = _run_parametric_route(
            long_data=long_data,
            analysis_data=analysis_data,
            subject=subject,
            treatments=treatments,
            dv_name=dv_name,
            alpha=alpha,
        )
    else:
        route = "non-parametric"
        inferential = _run_nonparametric_route(
            analysis_data=analysis_data,
            treatments=treatments,
            alpha=alpha,
        )

    posthoc = inferential["posthoc"]
    significant_posthoc = None
    if posthoc is not None:
        significant_posthoc = posthoc.loc[posthoc["significant"]].copy()

    # All comparisons against the reference, taken from the Holm-corrected
    # pairwise family (not filtered by significance).
    reference_posthoc = None
    if posthoc is not None and reference is not None:
        if reference not in treatments:
            raise ValueError(
                f"Reference treatment '{reference}' is not in treatments."
            )

        rows = []
        for _, row in posthoc.iterrows():
            if reference not in {row["A"], row["B"]}:
                continue

            reference_is_a = row["A"] == reference
            other = row["B"] if reference_is_a else row["A"]

            mean_reference = (
                float(row["mean_A"]) if reference_is_a
                else float(row["mean_B"])
            )
            mean_treatment = (
                float(row["mean_B"]) if reference_is_a
                else float(row["mean_A"])
            )

            if route == "parametric":
                statistic_name = "t"
                statistic = float(row["T"])
                if reference_is_a:
                    statistic = -statistic
                    effect = -float(row["hedges"])
                else:
                    effect = float(row["hedges"])
                effect_name = "Hedges' g_av"
            else:
                statistic_name = "W"
                statistic = float(row["W"])
                effect = (
                    -float(row["RBC"])
                    if reference_is_a
                    else float(row["RBC"])
                )
                effect_name = "Rank-biserial correlation"

            rows.append({
                "Treatment": other,
                "Mean treatment": mean_treatment,
                "Mean reference": mean_reference,
                "Delta vs. reference": mean_treatment - mean_reference,
                "Statistic name": statistic_name,
                "Statistic": statistic,
                "p_unc": float(row["p_unc"]),
                "p_corr": float(row["p_corr"]),
                "Effect name": effect_name,
                "Effect vs. reference": effect,
                "significant": bool(row["significant"]),
            })

        reference_posthoc = (
            pd.DataFrame(rows)
            .sort_values(
                ["Mean treatment", "Treatment"],
                ascending=[False, True],
            )
            .reset_index(drop=True)
        )

    print("DESCRIPTIVE STATISTICS")
    display(descriptive.round({"Mean": 4, "SD": 4}))

    print("SHAPIRO-WILK TESTS")
    display(shapiro.round({"W": 4, "p": 4}))

    print(f"SELECTED ROUTE: {route}")

    if inferential["sphericity"] is not None:
        sphericity = inferential["sphericity"]
        display(pd.DataFrame([{
            "Mauchly W": sphericity.W,
            "chi-square": sphericity.chi2,
            "df": sphericity.dof,
            "p": sphericity.pval,
            "Sphericity": bool(sphericity.spher),
        }]).round(4))

    display(pd.DataFrame([inferential["omnibus"]]).round(4))

    if posthoc is None:
        print(
            "The omnibus test was not significant; "
            "post-hoc comparisons were not performed."
        )
    elif significant_posthoc.empty:
        print(
            "No pairwise comparison remained significant after Holm correction."
        )
    else:
        print(
            "SIGNIFICANT HOLM-CORRECTED COMPARISONS "
            f"({len(significant_posthoc)} of {len(posthoc)})"
        )
        display(significant_posthoc.round(4))

    if reference_posthoc is not None:
        reference_label = _pretty_label(reference)
        print(f"ALL PAIRWISE COMPARISONS AGAINST THE REFERENCE ({reference_label})")
        display(reference_posthoc.round(4))
        n_significant_reference = int(reference_posthoc["significant"].sum())
        print(
            f"{n_significant_reference} of {len(reference_posthoc)} "
            "comparisons against the reference remained significant "
            "after Holm correction."
        )

    return {
        "route": route,
        "analysis_data": analysis_data,
        "long_data": long_data,
        "descriptive": descriptive,
        "shapiro": shapiro,
        "sphericity": inferential["sphericity"],
        "anova": inferential["anova"],
        "omnibus": inferential["omnibus"],
        "posthoc": posthoc,
        "significant_posthoc": significant_posthoc,
        "reference_posthoc": reference_posthoc,
        "reference": reference,
        "alpha": alpha,
    }


def analyze_factor(factor_key):
    cfg = FACTORS[factor_key]
    df = load_factor_data(factor_key)

    results = run_repeated_measures_analysis(
        data=df,
        subject=SUBJECT,
        treatments=cfg["treatments"],
        reference=cfg["reference"],
    )

    return df, results


## Factor 1 — Class-definition enrichment

In [3]:
factor_1_definitions_data, factor_1_definitions_results = analyze_factor("factor_1_definitions")

Data file: results\rq3_2\metrics\factor_1_macro_f1.csv
Projects: 30
Analyzed treatments (5): ['p0_baseline', 'p1_definition_augmented', 'p2_angular_definition', 'p3_extensive_definition', 'p4_mix_definition']
DESCRIPTIVE STATISTICS


,N,Mean,SD
treatment,,,
p0_baseline,30,0.4850,0.0972
p2_angular_definition,30,0.4848,0.0932
p4_mix_definition,30,0.4691,0.0809
p1_definition_augmented,30,0.4650,0.0794
p3_extensive_definition,30,0.4476,0.0797


SHAPIRO-WILK TESTS


,N,W,p,Normality not rejected
Treatment,,,,
p0_baseline,30,0.9493,0.1615,True
p1_definition_augmented,30,0.9618,0.3450,True
p2_angular_definition,30,0.9381,0.0810,True
p3_extensive_definition,30,0.9722,0.6015,True
p4_mix_definition,30,0.9678,0.4815,True


SELECTED ROUTE: parametric


...

SIGNIFICANT HOLM-CORRECTED COMPARISONS (4 of 10)


,A,B,mean_A,mean_B,T,dof,p_unc,p_corr,p_adjust,hedges,significant
2,p0_baseline,p3_extensive_definition,0.4850,0.4476,3.0669,29.0,0.0047,0.0326,holm,0.4160,True
4,p1_definition_augmented,p2_angular_definition,0.4650,0.4848,-3.8987,29.0,0.0005,0.0047,holm,-0.2266,True
7,p2_angular_definition,p3_extensive_definition,0.4848,0.4476,4.7518,29.0,0.0001,0.0005,holm,0.4240,True
9,p3_extensive_definition,p4_mix_definition,0.4476,0.4691,-3.4388,29.0,0.0018,0.0143,holm,-0.2647,True


ALL PAIRWISE COMPARISONS AGAINST THE REFERENCE (Zero-shot baseline)


,Treatment,Mean treatment,Mean reference,Delta vs. reference,Statistic name,Statistic,p_unc,p_corr,Effect name,Effect vs. reference,significant
0,p2_angular_definition,0.4848,0.485,-0.0002,t,-0.0239,0.9811,0.9811,Hedges' g_av,-0.0022,False
1,p4_mix_definition,0.4691,0.485,-0.0159,t,-1.5057,0.1429,0.4288,Hedges' g_av,-0.1757,False
2,p1_definition_augmented,0.4650,0.485,-0.0201,t,-2.1552,0.0396,0.1583,Hedges' g_av,-0.2234,False
3,p3_extensive_definition,0.4476,0.485,-0.0374,t,-3.0669,0.0047,0.0326,Hedges' g_av,-0.4160,True


1 of 4 comparisons against the reference remained significant after Holm correction.


In [4]:
ph = factor_1_definitions_results["posthoc"]
ref = "p0_baseline"
print(ph.loc[(ph["A"] == ref) | (ph["B"] == ref), ["A", "B", "p_corr", "hedges"]].round(4))

             A                        B  p_corr  hedges
0  p0_baseline  p1_definition_augmented  0.1583  0.2234
1  p0_baseline    p2_angular_definition  0.9811  0.0022
2  p0_baseline  p3_extensive_definition  0.0326  0.4160
3  p0_baseline        p4_mix_definition  0.4288  0.1757


## Factor 2 — Heuristic guidance for ambiguous commits

In [5]:
factor_2_heuristics_data, factor_2_heuristics_results = analyze_factor("factor_2_heuristics")

Data file: results\rq3_2\metrics\factor_2_macro_f1.csv
Projects: 30
Analyzed treatments (12): ['p0_baseline', 'p2_angular_definition', 'p_negative_constraints', 'p_chore_rule_and_negative', 'p_chore_rule', 'p_file_pattern', 'p_priority_rules', 'p_chore_negative_and_dominant', 'p_header_structured', 'p_code_disambiguation', 'p_infra_disambiguation', 'p_disambiguation']


DESCRIPTIVE STATISTICS


,N,Mean,SD
treatment,,,
p_code_disambiguation,30,0.5030,0.0894
p_header_structured,30,0.4917,0.0930
p_chore_rule,30,0.4876,0.0981
p_chore_negative_and_dominant,30,0.4874,0.0952
p_chore_rule_and_negative,30,0.4869,0.0924
p0_baseline,30,0.4850,0.0972
p2_angular_definition,30,0.4848,0.0932
p_disambiguation,30,0.4829,0.0891
p_negative_constraints,30,0.4769,0.0884


SHAPIRO-WILK TESTS


,N,W,p,Normality not rejected
Treatment,,,,
p0_baseline,30,0.9493,0.1615,True
p2_angular_definition,30,0.9381,0.0810,True
p_negative_constraints,30,0.9680,0.4863,True
p_chore_rule_and_negative,30,0.9607,0.3226,True
p_chore_rule,30,0.9549,0.2289,True
p_file_pattern,30,0.9748,0.6772,True
p_priority_rules,30,0.9581,0.2764,True
p_chore_negative_and_dominant,30,0.9434,0.1126,True
p_header_structured,30,0.9641,0.3915,True


...

SIGNIFICANT HOLM-CORRECTED COMPARISONS (9 of 66)


,A,B,mean_A,mean_B,T,dof,p_unc,p_corr,p_adjust,hedges,significant
20,p2_angular_definition,p_priority_rules,0.4848,0.4498,4.9360,29.0,0.0000,0.0019,holm,0.4037,True
29,p_chore_negative_and_dominant,p_priority_rules,0.4874,0.4498,4.9065,29.0,0.0000,0.0020,holm,0.4279,True
37,p_chore_rule,p_priority_rules,0.4876,0.4498,4.5536,29.0,0.0001,0.0053,holm,0.4221,True
44,p_chore_rule_and_negative,p_priority_rules,0.4869,0.4498,4.6579,29.0,0.0001,0.0040,holm,0.4299,True
46,p_code_disambiguation,p_file_pattern,0.5030,0.4656,5.7873,29.0,0.0000,0.0002,holm,0.4362,True
49,p_code_disambiguation,p_negative_constraints,0.5030,0.4769,3.8288,29.0,0.0006,0.0369,holm,0.2896,True
50,p_code_disambiguation,p_priority_rules,0.5030,0.4498,8.7561,29.0,0.0000,0.0000,holm,0.6273,True
62,p_header_structured,p_priority_rules,0.4917,0.4498,5.4534,29.0,0.0000,0.0005,holm,0.4835,True
65,p_negative_constraints,p_priority_rules,0.4769,0.4498,3.9961,29.0,0.0004,0.0239,holm,0.3219,True


ALL PAIRWISE COMPARISONS AGAINST THE REFERENCE (Zero-shot baseline)


,Treatment,Mean treatment,Mean reference,Delta vs. reference,Statistic name,Statistic,p_unc,p_corr,Effect name,Effect vs. reference,significant
0,p_code_disambiguation,0.5030,0.485,0.0179,t,1.5686,0.1276,1.0000,Hedges' g_av,0.1895,False
1,p_header_structured,0.4917,0.485,0.0067,t,0.6308,0.5331,1.0000,Hedges' g_av,0.0692,False
2,p_chore_rule,0.4876,0.485,0.0025,t,0.2935,0.7712,1.0000,Hedges' g_av,0.0256,False
3,p_chore_negative_and_dominant,0.4874,0.485,0.0024,t,0.2753,0.7850,1.0000,Hedges' g_av,0.0243,False
4,p_chore_rule_and_negative,0.4869,0.485,0.0019,t,0.1865,0.8533,1.0000,Hedges' g_av,0.0195,False
5,p2_angular_definition,0.4848,0.485,-0.0002,t,-0.0239,0.9811,1.0000,Hedges' g_av,-0.0022,False
6,p_disambiguation,0.4829,0.485,-0.0021,t,-0.1341,0.8943,1.0000,Hedges' g_av,-0.0225,False
7,p_negative_constraints,0.4769,0.485,-0.0082,t,-0.7357,0.4678,1.0000,Hedges' g_av,-0.0867,False
8,p_infra_disambiguation,0.4741,0.485,-0.0109,t,-0.6599,0.5145,1.0000,Hedges' g_av,-0.1180,False
9,p_file_pattern,0.4656,0.485,-0.0195,t,-1.5413,0.1341,1.0000,Hedges' g_av,-0.2165,False


0 of 11 comparisons against the reference remained significant after Holm correction.


## Factor 3 — Reasoning and example-based prompting

In [6]:
factor_3_few_shot_cot_data, factor_3_few_shot_cot_results = analyze_factor("factor_3_few_shot_cot")

Data file: results\rq3_2\metrics\factor_3_macro_f1.csv
Projects: 30
Analyzed treatments (5): ['p0_baseline', 'p_code_disambiguation', 'p_zero-shot_cot', 'p_few-shot', 'p_cot_few-shot']
DESCRIPTIVE STATISTICS


,N,Mean,SD
treatment,,,
p_code_disambiguation,30,0.5030,0.0894
p_zero-shot_cot,30,0.5001,0.0892
p_cot_few-shot,30,0.4972,0.0880
p_few-shot,30,0.4911,0.0844
p0_baseline,30,0.4850,0.0972


SHAPIRO-WILK TESTS


,N,W,p,Normality not rejected
Treatment,,,,
p0_baseline,30,0.9493,0.1615,True
p_code_disambiguation,30,0.9677,0.4787,True
p_zero-shot_cot,30,0.9805,0.8385,True
p_few-shot,30,0.9747,0.6753,True
p_cot_few-shot,30,0.9744,0.6665,True


SELECTED ROUTE: parametric


,Mauchly W,chi-square,df,p,Sphericity
0,0.1959,44.6939,9,0.0,False


,test,statistic_name,statistic,df1,df2,p,effect_name,effect,epsilon,correction,significant
0,Repeated-measures ANOVA,F,1.3337,2.2775,66.0464,0.2715,eta_G_squared,0.0053,0.5694,Greenhouse-Geisser,False


The omnibus test was not significant; post-hoc comparisons were not performed.


## Factor 4 — Confidence-based abstention

In [7]:
factor_4_abstention_data, factor_4_abstention_results = analyze_factor("factor_4_abstention")

Data file: results\rq3_2\metrics\factor_4_macro_f1.csv
Projects: 30
Analyzed treatments (4): ['p0_baseline', 'p_code_disambiguation', 'p_exclude_low', 'p_high_only']
DESCRIPTIVE STATISTICS


,N,Mean,SD
treatment,,,
p_code_disambiguation,30,0.5030,0.0894
p_high_only,30,0.4907,0.0969
p0_baseline,30,0.4850,0.0972
p_exclude_low,30,0.4803,0.0906


SHAPIRO-WILK TESTS


,N,W,p,Normality not rejected
Treatment,,,,
p0_baseline,30,0.9493,0.1615,True
p_code_disambiguation,30,0.9677,0.4787,True
p_exclude_low,30,0.9625,0.3583,True
p_high_only,30,0.9704,0.5498,True


SELECTED ROUTE: parametric


,Mauchly W,chi-square,df,p,Sphericity
0,0.2471,38.7492,5,0.0,False


,test,statistic_name,statistic,df1,df2,p,effect_name,effect,epsilon,correction,significant
0,Repeated-measures ANOVA,F,2.3636,1.6102,46.6959,0.1153,eta_G_squared,0.0084,0.5367,Greenhouse-Geisser,False


The omnibus test was not significant; post-hoc comparisons were not performed.


## Cross-factor summary

In [8]:
FACTOR_RESULTS = {
    "factor_1_definitions": factor_1_definitions_results,
    "factor_2_heuristics": factor_2_heuristics_results,
    "factor_3_few_shot_cot": factor_3_few_shot_cot_results,
    "factor_4_abstention": factor_4_abstention_results,
}

baseline_means = [
    results["descriptive"].loc["p0_baseline", "Mean"]
    for results in FACTOR_RESULTS.values()
]

if not np.allclose(baseline_means, baseline_means[0], atol=1e-12, rtol=0):
    raise RuntimeError(
        f"Baseline means differ across factor CSVs: {baseline_means}"
    )

baseline_mean = float(baseline_means[0])

CHECKPOINTS = [
    ("Baseline", "factor_1_definitions", "p0_baseline"),
    ("Definitions", "factor_1_definitions", "p2_angular_definition"),
    ("Heuristics", "factor_2_heuristics", "p_code_disambiguation"),
    ("Reasoning", "factor_3_few_shot_cot", "p_zero-shot_cot"),
    ("Examples", "factor_3_few_shot_cot", "p_few-shot"),
    ("Combined", "factor_3_few_shot_cot", "p_cot_few-shot"),
    ("Abstention", "factor_4_abstention", "p_exclude_low"),
    ("Abstention", "factor_4_abstention", "p_high_only"),
]

rows = []

for checkpoint, factor_key, treatment in CHECKPOINTS:
    descriptive = FACTOR_RESULTS[factor_key]["descriptive"]
    row = descriptive.loc[treatment]

    rows.append({
        "Checkpoint": checkpoint,
        "Treatment": _pretty_label(treatment),
        "Mean Macro-F1": float(row["Mean"]),
        "SD": float(row["SD"]),
        "Delta vs. baseline": (
            np.nan
            if treatment == "p0_baseline"
            else float(row["Mean"] - baseline_mean)
        ),
    })

checkpoint_summary = pd.DataFrame(rows)

formatted = checkpoint_summary.copy()
formatted["Mean Macro-F1"] = formatted["Mean Macro-F1"].map(lambda x: f"{x:.3f}")
formatted["SD"] = formatted["SD"].map(lambda x: f"{x:.3f}")
formatted["Delta vs. baseline"] = formatted["Delta vs. baseline"].map(
    lambda x: "---" if pd.isna(x) else f"{x:+.3f}"
)

display(formatted)


,Checkpoint,Treatment,Mean Macro-F1,SD,Delta vs. baseline
0,Baseline,Zero-shot baseline,0.485,0.097,---
1,Definitions,Angular definitions,0.485,0.093,-0.000
2,Heuristics,Code disambiguation,0.503,0.089,+0.018
3,Reasoning,Zero-shot chain-of-thought,0.500,0.089,+0.015
4,Examples,Few-shot,0.491,0.084,+0.006
5,Combined,Few-shot + chain-of-thought,0.497,0.088,+0.012
6,Abstention,Exclude low confidence,0.480,0.091,-0.005
7,Abstention,High confidence only,0.491,0.097,+0.006
